# 配套实践 12-02：条件 Flow Matching 沿速度场运输样本

本练习使用与实践 12-01 相同的条件双峰动作分布。训练时在 Gaussian 噪声和真实动作之间抽取线性插值点，监督网络预测配对端点的速度；生成时从噪声出发，用 Euler 方法积分学习到的条件速度场。依赖：PyTorch、NumPy、Matplotlib；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/intermediate/12-diffusion-and-flow-matching/" target="_blank">在新标签页返回课程正文</a>

In [ ]:
import math  # 计算连续生成时间的周期 embedding
import numpy as np  # 平滑训练曲线并整理绘图数据
import torch  # 构造概率路径并训练条件速度场
from torch import nn  # 使用多层感知机和激活函数
import matplotlib.pyplot as plt  # 绘制配对路径、训练损失和 ODE 积分
torch.set_num_threads(2)  # 限制轻量实验的 CPU 线程开销
torch.manual_seed(13)  # 固定训练配对、模型初始化与生成噪声
np.random.seed(13)  # 固定 NumPy 侧随机过程
plt.rcParams["figure.dpi"] = 120  # 提高笔记本图像显示清晰度

## 1. 在线性概率路径上构造监督

噪声端记为 x0，数据端记为 x1。对随机生成时间 s，训练点是两端的线性插值，目标速度是 x1−x0。下图只画少量配对，用线段展示单个训练目标；网络最终学习的是大量随机配对形成的平均条件速度场。

In [ ]:
def sample_action_data(sample_count):  # 定义与 Diffusion 实验一致的条件双峰动作分布
    contexts = torch.randint(0, 2, (sample_count,))  # 随机选择左侧或右侧任务 Context
    mode_signs = torch.where(torch.rand(sample_count) > 0.5, 1.0, -1.0)  # 在每个 Context 中随机选择上下动作模式
    horizontal_centers = torch.where(contexts == 0, -1.5, 1.5)  # 根据 Context 决定数据横向中心
    centers = torch.stack([horizontal_centers, mode_signs], dim=1)  # 组合四个条件模式中心
    actions = centers + 0.12 * torch.randn(sample_count, 2)  # 加入小幅示范变化形成连续数据分布
    return actions, contexts  # 返回二维动作与条件标签
data_endpoints, path_contexts = sample_action_data(28)  # 采样少量真实动作作为路径终点
noise_endpoints = torch.randn_like(data_endpoints)  # 为每个真实动作独立采样 Gaussian 起点
shown_time = 0.45  # 选择一个中间生成时间展示插值点
intermediate_points = (1.0 - shown_time) * noise_endpoints + shown_time * data_endpoints  # 计算线性概率路径上的中间位置
fig, axis = plt.subplots(figsize=(8.4, 5.0))  # 创建噪声到数据的配对路径图
for sample_index in range(len(data_endpoints)):  # 逐条绘制随机噪声与数据动作的配对
    color = "#2563eb" if path_contexts[sample_index] == 0 else "#ea580c"  # 使用颜色区分两个任务 Context
    axis.plot([noise_endpoints[sample_index, 0], data_endpoints[sample_index, 0]], [noise_endpoints[sample_index, 1], data_endpoints[sample_index, 1]], color=color, alpha=0.22)  # 绘制该配对的直线路径
axis.scatter(noise_endpoints[:, 0], noise_endpoints[:, 1], color="#94a3b8", s=28, label="Noise x0")  # 显示 Gaussian 路径起点
axis.scatter(intermediate_points[:, 0], intermediate_points[:, 1], color="#16a34a", s=34, label=f"Interpolated xs, s={shown_time}")  # 显示随机路径中的训练位置
axis.scatter(data_endpoints[:, 0], data_endpoints[:, 1], c=np.where(path_contexts.numpy() == 0, "#2563eb", "#ea580c"), s=45, label="Data x1")  # 显示条件动作终点
axis.set(title="Flow Matching supervises velocities along a probability path", xlabel="Action dim 1", ylabel="Action dim 2", xlim=(-3.0, 3.0), ylim=(-2.7, 2.7))  # 标注概率路径及动作坐标
axis.legend()  # 显示噪声、中间点和数据端点图例
axis.grid(alpha=0.2)  # 添加淡网格帮助观察运输方向
fig.tight_layout()  # 调整图像边距
plt.show()  # 显示 Flow Matching 训练样本的构造

**怎样理解结果：** 灰点是噪声起点，蓝/橙点是两个 Context 的真实动作，绿点位于二者之间。每条线的目标速度都指向配对数据点，但网络并不知道完整线段，只接收一个中间位置、生成时间和 Context。大量随机配对的回归结果共同形成可用于运输整个分布的速度场。

## 2. 学习条件速度场

速度网络与上一实践的噪声网络具有相同规模，也接收 7 维输入。区别在监督：这里直接回归线性路径速度 x1−x0，不使用噪声日程或反向后验公式。

In [ ]:
class VelocityField(nn.Module):  # 定义连续时间条件速度场网络
    def __init__(self):  # 初始化三层小型感知机
        super().__init__()  # 初始化 PyTorch 模型基类
        self.network = nn.Sequential(nn.Linear(7, 64), nn.SiLU(), nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, 2))  # 把动作、时间和 Context 映射到二维速度
    def forward(self, positions, times, contexts):  # 定义概率路径任意位置的条件速度预测
        time_features = torch.cat([times, torch.sin(2.0 * math.pi * times), torch.cos(2.0 * math.pi * times)], dim=1)  # 建立连续生成时间的线性与周期特征
        context_features = nn.functional.one_hot(contexts, num_classes=2).float()  # 把任务 Context 转成 one-hot 特征
        model_inputs = torch.cat([positions, time_features, context_features], dim=1)  # 拼接当前位置、生成时间和任务条件
        return self.network(model_inputs)  # 输出当前条件速度向量
velocity_field = VelocityField()  # 创建待训练的 Flow Matching 网络
optimizer = torch.optim.Adam(velocity_field.parameters(), lr=0.001)  # 使用 Adam 更新速度场参数
loss_history = []  # 保存每个训练批次的速度回归 MSE
for update_index in range(3000):  # 使用三千个轻量批次训练条件速度场
    data_batch, context_batch = sample_action_data(256)  # 采样概率路径的数据端动作
    noise_batch = torch.randn_like(data_batch)  # 独立采样标准 Gaussian 路径起点
    time_batch = torch.rand(256, 1)  # 为每个配对均匀采样连续生成时间
    path_batch = (1.0 - time_batch) * noise_batch + time_batch * data_batch  # 计算线性概率路径上的训练位置
    target_velocity = data_batch - noise_batch  # 计算当前配对在线性路径上的恒定目标速度
    predicted_velocity = velocity_field(path_batch, time_batch, context_batch)  # 根据位置、时间和条件预测局部速度
    loss = ((predicted_velocity - target_velocity) ** 2).mean()  # 计算预测速度与条件目标速度的均方误差
    optimizer.zero_grad()  # 清除上一个批次残留的梯度
    loss.backward()  # 反向传播计算速度场参数梯度
    optimizer.step()  # 更新网络使局部速度更接近训练目标
    loss_history.append(float(loss.detach()))  # 保存当前批次的速度回归损失
smoothed_loss = np.convolve(loss_history, np.ones(80) / 80.0, mode="valid")  # 对随机批次损失执行滑动平均
fig, axis = plt.subplots(figsize=(8.8, 3.6))  # 创建 Flow Matching 训练曲线
axis.plot(np.arange(len(smoothed_loss)) + 79, smoothed_loss, color="#7c3aed")  # 绘制平滑后的速度 MSE
axis.set(title="The network learns conditional velocity along random paths", xlabel="Update", ylabel="Velocity MSE")  # 标注训练步与速度误差
axis.grid(alpha=0.2)  # 添加淡网格帮助观察总体收敛
fig.tight_layout()  # 调整图像边距
plt.show()  # 显示条件速度场的学习过程

**怎样理解结果：** 损失明显下降但不会接近零，因为同一中间位置可能由许多噪声—数据配对经过，对应目标速度并不唯一。平方损失学习这些条件速度的平均，它产生的边际速度场仍可把整体噪声分布运输到数据分布。

## 3. 用 Euler 方法积分学习到的 ODE

生成时不再提供数据终点。我们只给 Gaussian 噪声、Context 和从 0 到 1 的时间网格，每一步沿网络速度移动一小段，并保存五个中间分布。

In [ ]:
@torch.no_grad()  # 关闭 ODE 生成过程的梯度记录
def integrate_with_snapshots(sample_count, context_value, integration_steps=50):  # 定义使用 Euler 求解条件流的函数
    current_actions = torch.randn(sample_count, 2)  # 从标准 Gaussian 采样初始动作粒子
    context_batch = torch.full((sample_count,), context_value, dtype=torch.long)  # 为全部粒子设置相同 Context
    time_step = 1.0 / integration_steps  # 计算每次 Euler 更新的生成时间间隔
    snapshot_indices = {0, 9, 24, 39, 49}  # 选择五个积分阶段用于可视化
    snapshots = {}  # 准备保存中间粒子分布
    for integration_index in range(integration_steps):  # 从生成时间零推进到一
        current_time = torch.full((sample_count, 1), integration_index / integration_steps)  # 为当前批次建立连续时间张量
        current_velocity = velocity_field(current_actions, current_time, context_batch)  # 读取当前位置的条件速度
        current_actions = current_actions + time_step * current_velocity  # 使用显式 Euler 方法前进一步
        if integration_index in snapshot_indices:  # 检查当前阶段是否需要保存
            snapshots[integration_index] = current_actions.clone()  # 保存不会受后续积分修改的粒子副本
    return snapshots  # 返回五个积分阶段的动作分布
left_snapshots = integrate_with_snapshots(700, 0)  # 将一组噪声运输到左侧条件分布
right_snapshots = integrate_with_snapshots(700, 1)  # 将另一组噪声运输到右侧条件分布
display_indices = [0, 9, 24, 39, 49]  # 按 ODE 正向积分顺序排列快照
fig, axes = plt.subplots(1, 5, figsize=(14, 3.1), sharex=True, sharey=True)  # 创建五幅运输过程图
for axis, integration_index in zip(axes, display_indices):  # 依次绘制每个积分阶段
    left_points = left_snapshots[integration_index]  # 读取左 Context 的当前粒子位置
    right_points = right_snapshots[integration_index]  # 读取右 Context 的当前粒子位置
    axis.scatter(left_points[:, 0], left_points[:, 1], s=7, alpha=0.28, color="#2563eb", label="Context 0")  # 绘制蓝色条件粒子
    axis.scatter(right_points[:, 0], right_points[:, 1], s=7, alpha=0.28, color="#ea580c", label="Context 1")  # 绘制橙色条件粒子
    axis.set(title=f"Euler step {integration_index + 1}", xlabel="Action dim 1", xlim=(-3.2, 3.2), ylim=(-2.8, 2.8))  # 标注当前积分步与动作范围
axes[0].set_ylabel("Action dim 2")  # 为共用纵轴标记第二动作维度
axes[-1].legend(loc="upper right", fontsize=8)  # 在最终图显示两个 Context 的颜色图例
fig.suptitle("The learned ODE transports noise into conditional action modes")  # 强调数值积分产生条件双峰分布
fig.tight_layout()  # 调整五幅子图间距
plt.show()  # 显示 Flow Matching 的完整生成过程

**怎样理解结果：** 最初两组粒子都近似 Gaussian；积分早期主要按 Context 向左或向右移动，后期逐渐在纵向分成上下模式。最终分布与 Diffusion 实验相似，但每一步执行的是速度场 ODE 更新，而不是离散反向后验采样。

**本练习的结论：** Flow Matching 直接学习概率路径的速度，并通过数值求解器生成动作。这里使用 50 步 Euler 便于观察，并不代表实际系统必须使用 50 次网络调用；减少步数、更换求解器或改变概率路径都要重新评估分布质量与闭环任务效果。